# Baseline Model Training

This notebook develops baseline machine learning models for the fashion trend prediction system.

The objective is to evaluate multiple algorithms and identify suitable baseline models before performing advanced model optimization.

Two prediction tasks are addressed:

**Trend Classification**
Predicting the trend category (`trend_label`) of a clothing item.

**Demand Forecasting**
Predicting the next week's search demand (`next_week_search`).

The models trained in this notebook will provide initial performance benchmarks for the system.


## Import Required Libraries

Machine learning experiments require several libraries for data handling, preprocessing, model training, and evaluation.

In this section, the necessary Python libraries such as **pandas**, **NumPy**, and **scikit-learn** are imported to support dataset manipulation, model training, and performance evaluation.


In [62]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

## Load Engineered Dataset

The dataset used for modeling was produced during the feature engineering stage.
It contains the original variables along with engineered temporal features such as lag variables and rolling statistics.

This dataset will be loaded and used for training and evaluating the baseline machine learning models.


In [63]:
df = pd.read_csv("../data/processed_data/fashion_trend_engineered.csv")

print(df.shape)

df.head()

(1416, 37)


,week,year,month,search_index,search_growth,momentum_4w,keyword_frequency,brand_popularity,avg_price,avg_rating,...,season_Monsoon,season_Summer,season_Winter,category_Jeans,category_Kurti,category_Saree,category_Streetwear,category_TShirt,trend_label,next_week_search
0,4,2021,1,55.04,0.125793,52.5100,153,0.54,1970.19,3.29,...,0,0,1,0,0,0,0,0,2,56.72
1,5,2021,2,56.72,0.030523,53.8450,150,0.46,1758.52,4.40,...,0,0,1,0,0,0,0,0,1,62.65
2,6,2021,2,62.65,0.104549,55.8250,189,0.89,2190.03,4.72,...,0,0,1,0,0,0,0,0,2,58.93
3,7,2021,2,58.93,-0.059377,58.3350,186,0.46,1803.50,4.03,...,0,0,1,0,0,0,0,0,0,64.61
4,8,2021,2,64.61,0.096386,60.7275,201,0.84,1884.70,3.32,...,0,0,1,0,0,0,0,0,2,63.18


## Data Integrity Check

Before training machine learning models, a quick validation step is performed to ensure that the dataset does not contain duplicate rows or unexpected inconsistencies.

Detecting duplicates helps confirm that the dataset represents unique observations and prevents biased model training caused by repeated records.


In [64]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## Prepare Features and Targets

The dataset contains two prediction targets:

* **trend_label** – a classification target representing the trend category.
* **next_week_search** – a regression target representing the expected search demand for the following week.

All remaining variables will be used as predictive features for training the machine learning models.


In [65]:
X = df.drop(columns=["trend_label", "next_week_search"])

y_class = df["trend_label"]
y_reg = df["next_week_search"]

print("Feature matrix shape:", X.shape)

Feature matrix shape: (1416, 35)


## Time-Based Train-Test Split

Since the dataset represents weekly time-series observations, a chronological split is used instead of a random split.

Earlier observations are used for training the models, while later observations are reserved for testing. This approach prevents information leakage from future data into the training process and better reflects real-world forecasting conditions.


In [66]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train_cls = y_class.iloc[:split_index]
y_test_cls = y_class.iloc[split_index:]

y_train_reg = y_reg.iloc[:split_index]
y_test_reg = y_reg.iloc[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1132, 35)
Test size: (284, 35)


## Feature Scaling

Some machine learning algorithms are sensitive to the scale of input variables.
For example, Logistic Regression and Support Vector Machines rely on distance-based optimization and perform better when features are standardized.

Feature scaling transforms the variables so that they have comparable magnitudes. Tree-based models such as Random Forest and Decision Trees do not require scaling because they split data using thresholds rather than distances.


In [67]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

## Classification Model Comparison

Multiple classification algorithms are trained to evaluate their ability to predict fashion trend categories.

The goal of this step is to compare different models and identify which algorithm provides the best predictive performance for the trend classification task.


In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [69]:
models_cls = {
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(),
    "DecisionTree": DecisionTreeClassifier()
}

In [70]:
results_cls = {}

for name, model in models_cls.items():

    if name in ["LogisticRegression", "SVM"]:
        model.fit(X_train_scaled, y_train_cls)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train_cls)
        preds = model.predict(X_test)

    acc = accuracy_score(y_test_cls, preds)
    results_cls[name] = acc

results_cls

{'LogisticRegression': 0.9683098591549296,
 'RandomForest': 1.0,
 'SVM': 0.8626760563380281,
 'DecisionTree': 1.0}

## Regression Model Comparison

In addition to trend classification, the system must also predict the expected search demand for the following week.

Several regression algorithms are trained and evaluated to determine which model best predicts the numerical demand value (`next_week_search`).


In [71]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

In [72]:
models_reg = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "SVR": SVR(),
    "DecisionTree": DecisionTreeRegressor()
}

In [73]:
results_reg = {}

for name, model in models_reg.items():

    if name == "SVR":
        model.fit(X_train_scaled, y_train_reg)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train_reg)
        preds = model.predict(X_test)

    mae = mean_absolute_error(y_test_reg, preds)
    results_reg[name] = mae

results_reg

{'LinearRegression': 2.907069343064973,
 'RandomForest': 3.554845422535215,
 'SVR': 5.153307999909536,
 'DecisionTree': 4.463838028169014}

## Baseline Model Performance

Multiple machine learning algorithms were evaluated for both classification and regression tasks.

For **trend classification**, Random Forest and Decision Tree models achieved the highest accuracy on the test dataset, indicating strong predictive capability for identifying fashion trend categories.

For **demand forecasting**, Linear Regression achieved the lowest mean absolute error, suggesting that the engineered temporal features provide strong linear predictive signals for estimating next week's search demand.

Based on these results:

• **Random Forest** is selected as the baseline classification model.
• **Linear Regression** is selected as the baseline regression model.

These models establish the initial performance benchmarks for the fashion trend prediction system and will serve as reference models for further optimization and evaluation.
